In [1]:
%%capture

%pip uninstall -y huggingface-hub
%pip install huggingface-hub>=1.5.0,<2.0

import re

import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
%pip install sentencepiece protobuf "datasets==4.3.0" hf_transfer
%pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install --no-deps "tokenizers>=0.22.0,<=0.23.0"
%pip install torchcodec

import torch; torch._dynamo.config.recompile_limit = 64;

%pip install --no-deps --upgrade timm  # For Gemma 4 vision/audio

%pip install unsloth optimum[onnxruntime] accelerate peft
%pip install --upgrade --no-cache-dir --no-deps unsloth transformers

import os
import gdown
import zipfile

In [2]:
from unsloth import FastLanguageModel
import torch

# Load your LoRA repo — unsloth automatically pulls the base model underneath
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="pameydorke/redred-gemma-4-E2B-it-lora",
    max_seq_length=1024,
    dtype=None,          # Auto-detect
    load_in_4bit=True,   # Load in 4-bit as it was trained
)

print("✅ Model + LoRA adapter loaded!")
print(f"Model class: {model.__class__.__name__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Exception: huggingface-hub>=1.5.0,<2.0 is required for a normal functioning of this module, but found huggingface-hub==0.36.2.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [6]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=1024,
    load_in_4bit=True,
)
model.load_adapter("pameydorke/redred-gemma-4-E2B-it-lora")

print("✅ Model + LoRA adapter loaded!")
print(f"Model class: {model.__class__.__name__}")

ValueError: `unsloth/gemma-4-e2b-it-unsloth-bnb-4bit` is not supported yet in `transformers==4.57.6`.
Please update transformers via `pip install --upgrade transformers` and try again.

In [ ]:
import os

merged_path = "/content/merged_model"
os.makedirs(merged_path, exist_ok=True)

# This merges the LoRA into the base weights and saves as a standard HF model
# "merged_16bit" = FP16 full weights (required for clean ONNX export)
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")

print(f"✅ Full merged model saved to: {merged_path}")